In [1]:
import torch 
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer 
from peft import get_peft_model, LoraConfig
from tqdm import tqdm
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

/home/omar-elmasaoudi/miniconda3/envs/ml_dev_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda


The CorDA tuning step looked at how to tune a CorDA model efficiently using hugging face's high level tuning library and training loops. In this notebook, I will demonstrate what happens under the hood when tuning a smaller model -- for testing purposes -- and as well as the effectiveness of adapter tuning on downstream tasks. 

### Load the gpt-neo-125M model 

In [14]:
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-125m")
model = AutoModelForCausalLM.from_pretrained("EleutherAI/gpt-neo-125m")
print(model)

GPTNeoForCausalLM(
  (transformer): GPTNeoModel(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(2048, 768)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPTNeoBlock(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPTNeoAttention(
          (attention): GPTNeoSelfAttention(
            (attn_dropout): Dropout(p=0.0, inplace=False)
            (resid_dropout): Dropout(p=0.0, inplace=False)
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=False)
            (q_proj): Linear(in_features=768, out_features=768, bias=False)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPTNeoMLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (c_proj): Linear(in_fe

In [9]:
GPT_NEO_BLOCK = model.transformer.h
attention_modules = []
for gpt_block in GPT_NEO_BLOCK:
    attention_modules.append(gpt_block.attn.attention)

#Example of the models attention head 
print(attention_modules[0])

GPTNeoSelfAttention(
  (attn_dropout): Dropout(p=0.0, inplace=False)
  (resid_dropout): Dropout(p=0.0, inplace=False)
  (k_proj): Linear(in_features=768, out_features=768, bias=False)
  (v_proj): Linear(in_features=768, out_features=768, bias=False)
  (q_proj): Linear(in_features=768, out_features=768, bias=False)
  (out_proj): Linear(in_features=768, out_features=768, bias=True)
)


In [ ]:
#We'll use an example attention head to demonstrate the Linear layers we're performing Lora on 
attention_1 = attention_modules[0]
k_proj = attention_1.k_proj
## Print the raw tensor of the K proj matrix and confirm it's shape 
print(k_proj.weight.shape)
print(k_proj.weight.dtype)

torch.Size([768, 768])
torch.float32


### Save model's old params locally

We'll save the params of each block's q, k, v projections 

In [7]:
import torch as pt
import os

EXPERIMENT_NUMBER = 0
proj_modules = ["k_proj", "q_proj", "v_proj"]

for idx, attention in enumerate(attention_modules):
    block_dir = f"../model/lora/base_model_attention_proj_weights/block_{idx}"
    os.makedirs(block_dir, exist_ok=True)
    pt.save(attention.k_proj.weight.detach().cpu(), f"{block_dir}/k_proj.pt")
    pt.save(attention.q_proj.weight.detach().cpu(), f"{block_dir}/q_proj.pt")
    pt.save(attention.v_proj.weight.detach().cpu(), f"{block_dir}/v_proj.pt")


Let's now test a small dummy dataset to demonstrate the LoRa tuning process

In [14]:
ds = load_dataset("stanfordnlp/sst2", split="train")

### Take a Peak of a subset of the data 

In [16]:
for i in range(5):
    print(ds[i])

{'idx': 0, 'sentence': 'hide new secretions from the parental units ', 'label': 0}
{'idx': 1, 'sentence': 'contains no wit , only labored gags ', 'label': 0}
{'idx': 2, 'sentence': 'that loves its characters and communicates something rather beautiful about human nature ', 'label': 1}
{'idx': 3, 'sentence': 'remains utterly satisfied to remain the same throughout ', 'label': 0}
{'idx': 4, 'sentence': 'on the worst revenge-of-the-nerds clichés the filmmakers could dredge up ', 'label': 0}


## Define the LoRa PEFT helper functions 

In [ ]:
class LoRa(nn.Module): 
   def __init__(self, base_linear, r=8, alpha=16, lora_dropout=0.0):
       super().__init__()
       ## In here we defined the frozen weights of the target modules W_0 
       self.base = base_linear
       # Ensure that auto_grad for base weights are turned off so gradients arent updated 
       for p in self.base.parameters():
           p.requires_grad = False 

### Save model params of tuned model

In [ ]:
import torch as pt
import os

EXPERIMENT_NUMBER = 0
proj_modules = ["k_proj", "q_proj", "v_proj"]

for idx, attention in enumerate(attention_modules):
    block_dir = f"../model/lora/experiment_{EXPERIMENT_NUMBER}/block_{idx}"
    os.makedirs(block_dir, exist_ok=True)
    pt.save(attention.k_proj.weight.detach().cpu(), f"{block_dir}/k_proj.pt")
    pt.save(attention.q_proj.weight.detach().cpu(), f"{block_dir}/q_proj.pt")
    pt.save(attention.v_proj.weight.detach().cpu(), f"{block_dir}/v_proj.pt")

### Define LoRa Initializer

In [18]:

class LoRaHelper(nn.Module):
    def __init__(self,og_weights, alpha:int , in_dims: int = 768, out_dims: int = 768, r:int = 4 ):
        super().__init__()
        self.W = og_weights
        self.alpha = alpha 
        self.r = r 
        #B has dims R^{d * r} 
        self.B = torch.zeros(out_dims, r)
        self.init_A(r = r , k = in_dims)
        #.Parameter subclasses the tensor class and tell us that these tensor are learnable params inside of the nn.Module
        self.A = nn.Parameter(self.A)
        self.B = nn.Parameter(self.B)
         
    def forward(self, x):
        delta_w = self.B @ self.A
        F = self.W(x)
        G = torch.matmul(delta_w, x) * (self.alpha / self.r)
        h = F + G  
        return h
        
    # A has dims R ^ {r * k }
    def init_A(self, r: int, k:int ):
        self.A = torch.empty(r, k)
        self.A = torch.nn.init.normal(self.A, mean=0.0, std=0.02)
        

### Create tuning loop